# Embedding Factory Test Notebook

This notebook demonstrates how to use the new Embedding Factory with OpenAI embeddings (default: `text-embedding-3-small`).

## Features Tested:
1. Creating embedders with different providers
2. Generating embeddings for single texts
3. Batch embedding
4. Getting model dimensions
5. Integration with Elasticsearch

## 1. Import Required Libraries

In [1]:
import sys
import os

# Add project root to path
sys.path.insert(0, os.path.abspath('../../..'))

from src.app.services.embedding_factory.embedding_factory import EmbeddingFactory
from src.app.core.config import settings
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("✅ Imports successful!")
print(f"Default embedding provider: {settings.EMBEDDING_PROVIDER}")
print(f"Default embedding model: {settings.EMBEDDING_MODEL}")
print(f"Expected dimensions: {settings.VECTOR_DIMENSIONS}")

✅ Imports successful!
Default embedding provider: openai
Default embedding model: text-embedding-3-small
Expected dimensions: 1536


## 2. Create OpenAI Embedder (Default)

Using the default `text-embedding-3-small` model from OpenAI.

In [2]:
# Create embedder with default model
embedder = EmbeddingFactory.create("openai", "text-embedding-3-small")

print("✅ Embedder created successfully!")
print(f"Model dimensions: {embedder.get_dimensions()}")

✅ Embedder created successfully!
Model dimensions: 1536


## 3. Test Single Text Embedding

Generate embeddings for a single piece of text.

In [3]:
# Test with a sample text
sample_text = "Physiotherapy helps patients recover from injuries and manage chronic pain."

embeddings = embedder.embed(sample_text)

print(f"✅ Generated embeddings for: '{sample_text}'")
print(f"Embedding dimensions: {len(embeddings)}")
print(f"First 10 values: {embeddings[:10]}")
print(f"Embedding type: {type(embeddings)}")

✅ Generated embeddings for: 'Physiotherapy helps patients recover from injuries and manage chronic pain.'
Embedding dimensions: 1536
First 10 values: [-0.017990121617913246, 0.04752446711063385, 0.018057500943541527, 0.03764225170016289, 0.02829906903207302, 0.015620636753737926, -0.041797276586294174, -0.003590725129470229, 0.029534345492720604, 0.025222105905413628]
Embedding type: <class 'list'>


## 4. Test Batch Embedding

Generate embeddings for multiple texts in a single API call (more efficient).

In [4]:
# Multiple texts to embed
texts = [
    "Physical therapy for knee pain",
    "Exercise prescription for shoulder mobility",
    "Manual therapy techniques for back pain"
]

# Batch embedding (OpenAI only)
batch_embeddings = embedder.embed_batch(texts)

print(f"✅ Generated embeddings for {len(texts)} texts")
print(f"Number of embedding vectors: {len(batch_embeddings)}")
print(f"Each vector has {len(batch_embeddings[0])} dimensions")

for i, text in enumerate(texts):
    print(f"\nText {i+1}: '{text}'")
    print(f"  First 5 values: {batch_embeddings[i][:5]}")

✅ Generated embeddings for 3 texts
Number of embedding vectors: 3
Each vector has 1536 dimensions

Text 1: 'Physical therapy for knee pain'
  First 5 values: [-0.04038907587528229, 0.007460038643330336, -0.03463700786232948, -0.003247219370678067, -0.020101092755794525]

Text 2: 'Exercise prescription for shoulder mobility'
  First 5 values: [0.03838292509317398, -0.01646582782268524, -0.0027707519475370646, -0.004564506467431784, 0.009521056897938251]

Text 3: 'Manual therapy techniques for back pain'
  First 5 values: [-0.00997986551374197, 0.03689558804035187, 0.049165111035108566, 0.0006475355476140976, -0.0158807672560215]


## 5. Compare Different Models

Test different OpenAI embedding models and their dimensions.

In [ ]:
# Test different models
models = [
    "text-embedding-3-small",
    "text-embedding-3-large",
    "text-embedding-ada-002"
]

print("Model Comparison:")
print("-" * 60)

for model in models:
    try:
        dims = EmbeddingFactory.get_model_dimensions("openai", model)
        print(f"✅ {model:30s} | {dims} dimensions")
    except Exception as e:
        print(f"❌ {model:30s} | Error: {e}")

# Note: Only test-embedding-3-small is actually created to avoid multiple API calls
print("\n💡 Note: Using text-embedding-3-small for best cost/performance balance")

## 6. Test Similarity Between Embeddings

Calculate cosine similarity between two embedded texts.

In [5]:
import numpy as np

def cosine_similarity(vec1, vec2):
    """Calculate cosine similarity between two vectors."""
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    return dot_product / (norm1 * norm2)

# Compare similar and dissimilar texts
text1 = "Physical therapy for knee rehabilitation"
text2 = "Knee injury physiotherapy treatment"  # Similar
text3 = "Making a chocolate cake recipe"  # Different

emb1 = embedder.embed(text1)
emb2 = embedder.embed(text2)
emb3 = embedder.embed(text3)

similarity_similar = cosine_similarity(emb1, emb2)
similarity_different = cosine_similarity(emb1, emb3)

print("Similarity Test:")
print("-" * 60)
print(f"Text 1: '{text1}'")
print(f"Text 2: '{text2}'")
print(f"Similarity: {similarity_similar:.4f} (Should be HIGH)\n")

print(f"Text 1: '{text1}'")
print(f"Text 3: '{text3}'")
print(f"Similarity: {similarity_different:.4f} (Should be LOW)")

Similarity Test:
------------------------------------------------------------
Text 1: 'Physical therapy for knee rehabilitation'
Text 2: 'Knee injury physiotherapy treatment'
Similarity: 0.7472 (Should be HIGH)

Text 1: 'Physical therapy for knee rehabilitation'
Text 3: 'Making a chocolate cake recipe'
Similarity: 0.0426 (Should be LOW)


## 7. Summary & Next Steps

The embedding factory is working correctly! Here's what we tested:

✅ **Created embedder** with OpenAI text-embedding-3-small  
✅ **Generated single embeddings** (1536 dimensions)  
✅ **Batch embeddings** for multiple texts  
✅ **Compared models** (small, large, ada-002)  
✅ **Tested similarity** between related and unrelated texts  

### Integration with Vector Stores

The embedding factory is already integrated with:
- **Elasticsearch** (`ElasticSearchService`)
- **OpenSearch** (`OpenSearchService`)

Both automatically use the configured embedding provider from `config.py`:
```python
EMBEDDING_PROVIDER = "openai"
EMBEDDING_MODEL = "text-embedding-3-small"
```

### Cost Comparison (per 1M tokens)
- `text-embedding-3-small`: **$0.02** ← Best value!
- `text-embedding-3-large`: $0.13
- `text-embedding-ada-002`: $0.10